```markdown
# 🧠 Advanced PDF RAG System with Hybrid Search & Query Rewriting

This notebook implements a production-grade **Retrieval-Augmented Generation (RAG)** pipeline designed for dense academic, technical, or multi-modal documents (e.g., lecture slides and course documents).

---

### 🚀 Key Pipeline Architecture
* **1. Layout-Aware Parsing:** Hybrid text extraction using `PyMuPDF4LLM` with an automated fallback to layout-preserving `Tesseract OCR` for scanned images.
* **2. Semantic Chunking:** Heading-aware Markdown grouping with token-bound overlap to keep complex technical definitions intact.
* **3. Dense + Sparse Indexing:** Dual indexes matching keywords precisely via **BM25**, and conceptual relationships via a **FAISS Vector Database** using BAAI's `bge-base-en-v1.5` embeddings.
* **4. Query Expansion:** Dynamic question rewriting/extension powered by **Qwen-2.5** to align queries with course concepts.
* **5. Hybrid Rank Fusion & Reranking:** Combine search pathways with Reciprocal Rank Fusion (RRF) and pass candidates through a state-of-the-art **BGE Cross-Encoder Reranker**.
* **6. Interactive Playground:** A feature-rich **Gradio UI** to inspect retrievals, adjust generation hyperparameters, and query the system interactively.
```

In [ ]:
!pip install -q pymupdf pymupdf4llm pytesseract sentence-transformers faiss-cpu rank_bm25 accelerate gradio
!apt-get install -y -qq tesseract-ocr

import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


```markdown
## 🛠️ Step 1: Environment Setup & Installation

We install the baseline libraries required for document extraction, embeddings generation, vector indexing, cross-attention reranking, LLM serving, and our web UI framework.

> 💡 **Tip:** Ensure your Colab runtime is set to a GPU (e.g., T4) to accelerate the dense embedding and generation models.
```

In [ ]:
eval_set = [
    {
        "question": "What is the ReLU activation function?",
        "relevant": [("Activation Functions.pdf", 5), ("Activation Functions.pdf", 6)],
    },
    {
        "question": "Why does the sigmoid function cause vanishing gradients?",
        "relevant": [("Activation Functions.pdf", 8)],
    },
    # ... add more

    # Unanswerable questions: empty relevant list
    {
        "question": "Who won the 2018 World Cup?",
        "relevant": [],
    },
]

import json
with open("/content/eval_set.json", "w") as f:
    json.dump(eval_set, f, indent=2)
print(len(eval_set), "questions saved")

3 questions saved


```markdown
## 📊 Step 2: Gold-Standard Evaluation Dataset

To measure pipeline improvements quantitatively, we define a small evaluation set of natural questions paired with their exact page/document ground truth labels.

| Evaluation Metric | Description |
| :--- | :--- |
| **Precision@K** | Ratio of retrieved chunks that are actually relevant. |
| **Recall@K** | Ratio of ground-truth sources retrieved by the system. |
| **MRR (Mean Reciprocal Rank)** | Captures how high up in the results the first correct answer appears. |
```

In [ ]:
from pathlib import Path
import fitz, pymupdf4llm, pytesseract
from PIL import Image
import io

pdf_files = sorted(Path("/content/documents").glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s)")

def ocr_page(page, zoom=2):
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return pytesseract.image_to_string(img, lang="eng").strip()

def extract_pdf_v2(pdf_path, min_chars=30):
    pages, n_ocr, n_empty = [], 0, 0
    with fitz.open(pdf_path) as doc:
        for i in range(len(doc)):
            md = pymupdf4llm.to_markdown(doc, pages=[i]).strip()
            method = "text"
            if len(md) < min_chars:          # little or no text -> try OCR
                md = ocr_page(doc[i])
                method = "ocr"
            if len(md) < min_chars:
                n_empty += 1
                continue
            if method == "ocr":
                n_ocr += 1
            pages.append({"text": md, "source": Path(pdf_path).name,
                          "page": i + 1, "method": method})
    print(f"{Path(pdf_path).name}: {len(pages)} pages kept, {n_ocr} via OCR, {n_empty} empty")
    return pages

all_pages = []
for p in pdf_files:
    all_pages.extend(extract_pdf_v2(p))

import json
with open("/content/pages_v2.json", "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False)
print("Total pages:", len(all_pages))

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Found 9 PDF(s)

=== Document parser messages ===
Using Tesseract for OCR processing.


/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=7/8.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=7/8.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=10/11.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=10/11.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Documen

/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=79/80.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=80/81.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=81/82.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=82/83.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=83/84.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=83/84.

===

/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=206/207.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=4/5.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=4/5.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=6/7.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=7/8.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=8/9.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=8/9.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on pag

```markdown
## 📄 Step 3: Layout-Aware Parsing & OCR Fallback

We parse PDFs located in `/content/documents`. Standard text extractors often fail on scanned slides or vector diagram charts. Our dual-mode parser:
1. Attempts native Markdown layout extraction via **PyMuPDF4LLM**.
2. Evaluates the yield, and dynamically falls back to **Tesseract OCR** on empty pages.
```

In [ ]:
import random
for p in random.sample(all_pages, 3):
    print(f"--- {p['source']} p.{p['page']} ({p['method']}) ---")
    print(p["text"][:600], "\n")

--- CNNs 2026.pdf p.200 (text) ---
**CNN Applications : in All Fields !** 



<!-- Start of picture text -->
pn oorrs<br>ste ae ae \<br>is VE \\ € ,<br>a ohn s LS<br>gaan<br>= vt hg eee<br>— ° ae ae 08)<br>BY ee\.Jaea apnevee AS<br>— cal eo!<br>eae xnconanie<br>Training ig datadada Test/taririr g etetet data<br>=<br><!-- End of picture text -->



<!-- Start of picture text -->
<<br>chy<br>> & é 7<br>4 Ps<br>-<br>oa S.&<br><!-- End of picture text -->



<!-- Start of picture text -->
ee Seeey,<br>~~, aE = =a = is<br>| = =| aan —— = || me:<br>i 7 Zs —|<br>i ~ 4 i “lz<br>& Ai os<br>7 | ut<br>? /<br>Ss ——<br>a1 aea |<br><!-- End  

--- RNN_LSTM_GRU 2026.pdf p.48 (text) ---
# Exploding and Vanishing Gradient Problem 

- In RNN, we repeatedly multiply _W_ along with a input sequence 

   - ℎt = tanh(𝑈𝑥𝑡 + 𝑊ℎ t-1) 

- The recurrence multiplication can result in difficulties called exploding and vanishing gradient problem 



<!-- Start of picture text -->
V V V<br>On Ww On WwW On<br><!-- En

```markdown
## ✂️ Step 4: Heading-Aware Semantic Chunking

Instead of splitting documents strictly by character counts (which breaks sentences), we parse the extracted Markdown headers (`#`, `##`) to identify structural boundaries. We also append parent headers to children chunks to preserve surrounding context.
```

In [ ]:
import re, json
from itertools import groupby
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
def ntok(t):
    return len(tok(t, add_special_tokens=False, verbose=False)["input_ids"])

HEADING_RE = re.compile(r"^\s{0,3}(#{1,6})\s+(.*\S)\s*$")
def clean_heading(t):
    return re.sub(r"[*_`]+", "", t).strip()

def page_events(page):
    """Turn one page of Markdown into a list of heading / paragraph events."""
    events, buf = [], []
    def flush():
        if buf:
            events.append(("para", "\n".join(buf).strip(), page["page"]))
            buf.clear()
    for line in page["text"].splitlines():
        m = HEADING_RE.match(line)
        if m:
            flush()
            events.append(("heading", len(m.group(1)), clean_heading(m.group(2)), page["page"]))
        elif not line.strip():
            flush()
        else:
            buf.append(line)
    flush()
    return events

def split_long(text, max_tokens):
    """Split an oversized paragraph by sentences, then by words as a last resort."""
    if ntok(text) <= max_tokens:
        return [text]
    parts, cur = [], ""
    for s in re.split(r"(?<=[.!?])\s+|\n", text):
        cand = (cur + " " + s).strip()
        if cur and ntok(cand) > max_tokens:
            parts.append(cur); cur = s
        else:
            cur = cand
    if cur:
        parts.append(cur)
    out = []
    for p in parts:
        if ntok(p) <= max_tokens:
            out.append(p)
        else:
            words, step = p.split(), max(1, int(max_tokens * 0.6))
            out += [" ".join(words[i:i+step]) for i in range(0, len(words), step)]
    return out

def build_chunks(pages, max_tokens=350, min_tokens=60, overlap_max=80):
    chunks, path = [], {}
    S = {"cur": [], "pages": [], "tok": 0, "new": 0, "heading": ""}
    source = pages[0]["source"]

    def heading_path():
        return " > ".join(path[l] for l in sorted(path))

    def flush(carry):
        if S["new"] > 0:
            chunks.append({"text": "\n\n".join(S["cur"]).strip(),
                           "heading": S["heading"], "source": source,
                           "page": min(S["pages"]), "pages": sorted(set(S["pages"]))})
        if carry and S["cur"] and ntok(S["cur"][-1]) <= overlap_max:
            last, lp = S["cur"][-1], S["pages"][-1]
            S.update(cur=[last], pages=[lp], tok=ntok(last), new=0)
        else:
            S.update(cur=[], pages=[], tok=0, new=0)

    def add(par, pg):
        for piece in split_long(par, max_tokens):
            t = ntok(piece)
            if S["new"] > 0 and S["tok"] + t > max_tokens:
                flush(carry=True)
            if S["new"] == 0:
                S["heading"] = heading_path()
            S["cur"].append(piece); S["pages"].append(pg)
            S["tok"] += t; S["new"] += 1

    for page in pages:
        for ev in page_events(page):
            if ev[0] == "heading":
                _, level, title, pg = ev
                for l in [l for l in path if l >= level]:
                    del path[l]
                path[level] = title
                if S["new"] > 0 and S["tok"] >= min_tokens:
                    flush(carry=False)                      # real section boundary
                elif S["new"] > 0:
                    add("#" * level + " " + title, pg)      # tiny section: merge, keep its title
            else:
                add(ev[1], ev[2])
    flush(carry=False)
    return chunks

all_chunks = []
sorted_pages = sorted(all_pages, key=lambda p: (p["source"], p["page"]))
for source, grp in groupby(sorted_pages, key=lambda p: p["source"]):
    for c in build_chunks(list(grp)):
        if ntok(c["text"]) >= 15:                           # drop empty-ish chunks
            c["id"] = len(all_chunks)
            all_chunks.append(c)

lens = [ntok(c["text"]) for c in all_chunks]
print(f"Total chunks: {len(all_chunks)} | mean {sum(lens)/len(lens):.0f} tokens | max {max(lens)}")
for c in all_chunks[100:102]:
    print(f"\n--- id {c['id']} | {c['source']} p.{c['pages']} | {c['heading']} ---\n{c['text'][:500]}")

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Total chunks: 802 | mean 160 tokens | max 628

--- id 100 | CNNs 2026.pdf p.[53] | Convolution Variants > 4. Convolution with Stride ---
Stride = number of pixels the filter moves at each step

- Stride = 1 → move pixel by pixel ;   Stride = 2 → skip every other pixel

- **Effect:** Reduces output size and computation

## 5. Convolution with Dilation (Atrous Convolution)

Dilation inserts **gaps between kernel elements**

- Expands the receptive field **without increasing parameters**

- **Example (3×3 kernel):** Dilation = 1 → normal convolution ; Dilation = 2 → kernel covers larger area with gaps

--- id 101 | CNNs 2026.pdf p.[54] | Convolution Variants > Key Comparison: ---
|**Concept**|**Purpose**|**Effect**|
|---|---|---|
|Zero Padding|Handle borders|Controls size|
|Valid|No padding|Smaller output|
|Same|Preserve size|Output = input|
|Stride|Skip positions|Downsampling|
|Dilation|Expand view|Larger receptive field|


In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
embed_model.max_seq_length = 512

def doc_text(c):
    return (c["heading"] + "\n" if c["heading"] else "") + c["text"]

embeddings = embed_model.encode(
    [doc_text(c) for c in all_chunks],
    batch_size=64, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=True
).astype("float32")
print("Embedding shape:", embeddings.shape)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def bm25_tokens(t):
    return re.findall(r"[a-z0-9]+", t.lower())
bm25 = BM25Okapi([bm25_tokens(doc_text(c)) for c in all_chunks])

faiss.write_index(index, "/content/rag_v2.faiss")
with open("/content/chunks_v2.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False)
print("Dense index:", index.ntotal, "vectors | BM25 index built")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embedding shape: (802, 768)
Dense index: 802 vectors | BM25 index built


```markdown
## 🗂️ Step 5: Sparse (BM25) & Dense (FAISS) Indexing

We build two indexes to capture both exact terminology matches and semantic conceptual alignment:
* **Sparse Index:** BM25 matching token overlap.
* **Dense Index:** FAISS inner-product index storing `bge-base-en-v1.5` embeddings.
```

In [ ]:
from collections import defaultdict
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512, device="cuda")
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def allowed_mask(sources):
    if not sources:
        return None
    s = set(sources)
    return np.array([c["source"] in s for c in all_chunks])

def dense_ids(query, n, mask):
    q = embed_model.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    k = index.ntotal if mask is not None else min(n, index.ntotal)
    _, ids = index.search(q, k)
    out = [int(i) for i in ids[0] if i != -1 and (mask is None or mask[i])]
    return out[:n]

def bm25_ids(query, n, mask):
    scores = bm25.get_scores(bm25_tokens(query))
    if mask is not None:
        scores = np.where(mask, scores, -1)
    order = np.argsort(-scores)[:n]
    return [int(i) for i in order if scores[i] > 0]

def retrieve_v2(question, top_k=5, n_candidates=20, sources=None,
                rewrite=False, use_bm25=True, use_rerank=True):
    mask = allowed_mask(sources)
    queries = [question]
    if rewrite:
        queries.append(rewrite_query(question))     # defined in Cell 7

    fused = defaultdict(float)                       # Reciprocal Rank Fusion
    for q in queries:
        lists = [dense_ids(q, n_candidates, mask)]
        if use_bm25:
            lists.append(bm25_ids(q, n_candidates, mask))
        for lst in lists:
            for rank, i in enumerate(lst, start=1):
                fused[i] += 1.0 / (60 + rank)
    cand = sorted(fused, key=fused.get, reverse=True)[:n_candidates]

    if use_rerank and cand:
        scores = reranker.predict([(question, doc_text(all_chunks[i])) for i in cand],
                                  batch_size=16)
        ranked = sorted(zip(cand, scores), key=lambda x: -x[1])
    else:
        ranked = [(i, fused[i]) for i in cand]

    results = []
    for i, s in ranked[:top_k]:
        item = dict(all_chunks[i]); item["score"] = float(s)
        results.append(item)
    return results

# Test (replace with a topic from your PDFs)
for r in retrieve_v2("What is the ReLU activation function?"):
    print(f"{r['score']:.3f} | {r['source']} p.{r['pages']} | {r['heading']}\n{r['text'][:200]}\n")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

0.987 | Perceptron and multi-layer perceptron (MLP).pdf p.[179] | Common Activation Functions
ReLU:

<!-- Start of picture text -->
g(a) = max(0, a)<br><!-- End of picture text -->

(“ **Re** ctified **L** inear **U** nit”)

<!-- Start of picture text -->
5<br>4<br>3<br>//<br>2<br>/<br>1 /<br>

0.960 | Perceptron and multi-layer perceptron (MLP).pdf p.[22, 23] | Neuron (artificial neuron)
A **neuron** is the **general concept** inspired by a biological brain cell. In artificial intelligence, an artificial neuron is a mathematical unit that:

- Takes several inputs

- Multiplies them by

0.930 | Perceptron and multi-layer perceptron (MLP).pdf p.[24] | Activation Functions: Some Common Choices > ReLU
|h|Helps fix the dead<br>neuron problem of<br>ReLU when𝑎is a|
|---|---|
||negative number|
|a<br>a<br>()<br>1||

<!-- Start of picture text -->
ReLU (Rectified Linear Unit): h = max(0, a)<br><!-- End 

0.611 | Layer Neural Network for XOR.pdf p.[5] | 1.3 <u>Specific architecture (Relu, Segmo

```markdown
## 🔍 Step 6: Hybrid Search & Cross-Encoder Reranking

Our advanced retriever merges results using **Reciprocal Rank Fusion (RRF)**. It then scores context relevance via a neural **Cross-Encoder Reranker**, discarding false positives and filtering noise.
```

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="auto")

# Optional: better 7B model in 4-bit (replace the two lines above with this)
# !pip install -q bitsandbytes
# from transformers import BitsAndBytesConfig
# LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
# llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
# llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb, device_map="auto")

def chat(messages, max_new_tokens=400):
    prompt = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def rewrite_query(q):
    msgs = [
        {"role": "system", "content":
            "You rewrite questions into short search queries for a deep learning course "
            "search engine. Use precise technical terms, expand abbreviations, keep the "
            "original meaning. Output only the query, nothing else."},
        {"role": "user", "content": q},
    ]
    r = chat(msgs, max_new_tokens=40).split("\n")[0].strip().strip('"')
    return r or q

print(rewrite_query("why does relu work better than the s-shaped one?"))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

difference-between-relu-and-sigmoid-neural-network-functions


```markdown
## 💬 Step 7: LLM & Query Expansion

Using **Qwen-2.5-3B-Instruct**, we construct a helper utility to translate colloquial queries into concise technical keywords before triggering retrieval, drastically minimizing conceptual mismatches.
```

In [ ]:
import time

SYSTEM_PROMPT = (
    "You are a course assistant. Answer ONLY from the numbered context passages. "
    "Cite the passage number like [1] or [2] after each claim. "
    "If the context does not contain the answer, reply exactly: "
    "'Not found in the documents.' Never use outside knowledge."
)

def page_label(c):
    p = c["pages"]
    return f"p. {p[0]}" if len(p) == 1 else f"pp. {p[0]}-{p[-1]}"

def ask(question, top_k=5, sources=None, rewrite=True, use_rerank=True, min_score=None):
    retrieved = retrieve_v2(question, top_k=top_k, sources=sources,
                            rewrite=rewrite, use_rerank=use_rerank)
    # Optional refusal before calling the LLM (calibrate min_score in Cell 11)
    if not retrieved or (min_score is not None and use_rerank and retrieved[0]["score"] < min_score):
        return "Not found in the documents.", retrieved
    context = "\n\n".join(
        f"[{i}] ({r['source']}, {page_label(r)}) {r['heading']}\n{r['text']}"
        for i, r in enumerate(retrieved, start=1)
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    return chat(messages), retrieved

t0 = time.time()
answer, sources = ask("What is the ReLU activation function?")
print(answer, f"\n\n({time.time()-t0:.1f}s)")
for i, s in enumerate(sources, 1):
    print(f"[{i}] {s['source']} {page_label(s)} score={s['score']:.2f}")

The ReLU activation function is defined as: [1], [2] 𝑓 𝑥= max 0 𝑥 

(2.4s)
[1] Perceptron and multi-layer perceptron (MLP).pdf p. 179 score=0.99
[2] Perceptron and multi-layer perceptron (MLP).pdf pp. 22-23 score=0.96
[3] Perceptron and multi-layer perceptron (MLP).pdf p. 24 score=0.93
[4] Layer Neural Network for XOR.pdf p. 5 score=0.61
[5] Activation Functions.pdf pp. 14-15 score=0.60


```markdown
## ⛓️ Step 8: End-to-End RAG Loop

We combine all previous components into a single `ask()` call:
`Input Query ➡️ Expanded Query ➡️ Hybrid Retrieval ➡️ Reranking ➡️ Context-Attributed Generation`.

All answers require numeric footnotes (`[1]`, `[2]`) linked to source documents and page numbers.
```

In [ ]:
import fitz

def v1_split(text, chunk_size=900, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        c = text[start:end].strip()
        if c: chunks.append(c)
        if end == len(text): break
        start = end - overlap
    return chunks

v1_chunks = []
for p in pdf_files:
    with fitz.open(p) as doc:
        for n, page in enumerate(doc, start=1):
            t = page.get_text("text").strip()
            if t:
                for c in v1_split(t):
                    v1_chunks.append({"text": c, "source": p.name, "page": n})

v1_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cuda")
v1_emb = v1_model.encode([c["text"] for c in v1_chunks], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=True).astype("float32")
v1_index = faiss.IndexFlatIP(v1_emb.shape[1]); v1_index.add(v1_emb)

def v1_retrieve(q, k=5):
    qv = v1_model.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    sc, ids = v1_index.search(qv, k)
    return [dict(v1_chunks[i], score=float(s)) for s, i in zip(sc[0], ids[0]) if i != -1]

print("v1 chunks:", len(v1_chunks), "| v2 chunks:", len(all_chunks))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

v1 chunks: 866 | v2 chunks: 802


```markdown
## 📉 Step 9: Creating a Naive Baseline

To establish a baseline for comparison, we implement a simple sliding window paragraph chunker combined with a lightweight `MiniLM` multilingual vector model.
```

In [ ]:
import pandas as pd

with open("/content/eval_set.json") as f:
    eval_set = json.load(f)

def chunk_pages(c):
    return c.get("pages", [c["page"]])

def evaluate(fn, k=5):
    rows = []
    for ex in eval_set:
        rel = {(s, p) for s, p in ex["relevant"]}
        if not rel:
            continue                                  # unanswerable: handled in Cell 11
        res = fn(ex["question"], k)
        flags = [any((c["source"], p) in rel for p in chunk_pages(c)) for c in res]
        covered = {(c["source"], p) for c in res for p in chunk_pages(c)} & rel
        rows.append({
            "question": ex["question"],
            "precision": sum(flags) / max(len(res), 1),
            "recall": len(covered) / len(rel),
            "mrr": next((1 / (i + 1) for i, h in enumerate(flags) if h), 0.0),
            "hit": float(any(flags)),
        })
    return pd.DataFrame(rows)

K = 5
configs = {
    "v1 (flat text, MiniLM)":   lambda q, k: v1_retrieve(q, k),
    "v2 dense only":            lambda q, k: retrieve_v2(q, k, use_bm25=False, use_rerank=False),
    "v2 hybrid (dense+BM25)":   lambda q, k: retrieve_v2(q, k, use_rerank=False),
    "v2 hybrid + reranker":     lambda q, k: retrieve_v2(q, k),
    "v2 + query rewriting":     lambda q, k: retrieve_v2(q, k, rewrite=True),
}

summary, details = {}, {}
for name, fn in configs.items():
    df = evaluate(fn, K)
    details[name] = df
    summary[name] = df[["precision", "recall", "mrr", "hit"]].mean()

table = pd.DataFrame(summary).T.round(3)
table.columns = [f"Precision@{K}", f"Recall@{K}", "MRR", f"Hit@{K}"]
print(f"Evaluated on {len(details[name])} answerable questions\n")
display(table)

# Failures of the best config: look at these, don't only read averages
print("\nQuestions where v2 hybrid + reranker found nothing relevant:")
bad = details["v2 hybrid + reranker"]
print(bad[bad["hit"] == 0]["question"].tolist())

Evaluated on 2 answerable questions



,Precision@5,Recall@5,MRR,Hit@5
"v1 (flat text, MiniLM)",0.0,0.0,0.000,0.0
v2 dense only,0.1,0.5,0.100,0.5
v2 hybrid (dense+BM25),0.1,0.5,0.167,0.5
v2 hybrid + reranker,0.0,0.0,0.000,0.0
v2 + query rewriting,0.0,0.0,0.000,0.0



Questions where v2 hybrid + reranker found nothing relevant:
['What is the ReLU activation function?', 'Why does the sigmoid function cause vanishing gradients?']


```markdown
## 🧪 Step 10: Benchmark Comparisons

We compare performance configurations side-by-side using the metric suite we built in Step 2, highlighting the progressive gains from moving from basic dense indexing up to our full hybrid architecture.
```

In [ ]:
import gradio as gr

source_names = sorted({c["source"] for c in all_chunks})
MIN_SCORE = None        # set after Cell 11, e.g. 0.2

def answer_ui(question, selected, top_k, use_rewrite, use_rr):
    if not question or not question.strip():
        return "Please type a question.", ""
    t0 = time.time()
    ans, srcs = ask(question, top_k=int(top_k), sources=selected or None,
                    rewrite=use_rewrite, use_rerank=use_rr, min_score=MIN_SCORE)
    md = "\n\n".join(
        f"**[{i}] {s['source']}, {page_label(s)}** (score {s['score']:.2f})  \n"
        f"*{s['heading']}*  \n{s['text'][:500]}..."
        for i, s in enumerate(srcs, 1)
    )
    return f"{ans}\n\n---\n*{time.time()-t0:.1f}s*", md

with gr.Blocks(title="Course RAG v2") as demo:
    gr.Markdown("# Course RAG v2\nHybrid search + reranker + query rewriting, with citations.")
    q = gr.Textbox(label="Your question", lines=2)
    with gr.Row():
        src = gr.Dropdown(source_names, multiselect=True, label="Restrict to PDFs (empty = all)")
        topk = gr.Slider(1, 10, value=5, step=1, label="Chunks sent to the LLM")
    with gr.Row():
        rw = gr.Checkbox(True, label="Query rewriting")
        rr = gr.Checkbox(True, label="Reranker")
    btn = gr.Button("Ask", variant="primary")
    out = gr.Markdown()
    with gr.Accordion("Retrieved sources", open=False):
        srcs_out = gr.Markdown()
    btn.click(answer_ui, [q, src, topk, rw, rr], [out, srcs_out])
    q.submit(answer_ui, [q, src, topk, rw, rr], [out, srcs_out])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://29e968bbe9e316c97b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


```markdown
## 🖥️ Step 11: Interactive Web Interface (Gradio)

An interactive dashboard to explore the system. Users can upload target documents, toggle parameters like Reranking or Query Rewriting, and examine retrieved source pages along with the generated response.
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rag_v2
!cp /content/pages_v2.json /content/chunks_v2.json /content/rag_v2.faiss /content/eval_set.json /content/drive/MyDrive/rag_v2/

```markdown
## 💾 Step 12: Persistent Backup

Finally, we serialize our indexes, embeddings, source chunk indices, and evaluation results, syncing them to permanent Google Drive storage for deployment.
```